# 使用 Meta 系列模型進行構建

## 簡介

本課程將涵蓋：

- 探索兩大主要的 Meta 系列模型－Llama 3.1 與 Llama 3.2
- 理解各模型的使用案例與場景
- 以程式碼範例展示各模型的獨特功能


## Meta 系列模型

在本課程中，我們將探討 Meta 系列或稱「Llama 群」的兩個模型－Llama 3.1 與 Llama 3.2

這些模型有不同的變體，並可於 [Microsoft Foundry Models catalog](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst) 取得。

> **注意：** GitHub Models 將於 2026 年 7 月底退役。詳情請參閱使用 [Microsoft Foundry Models](https://learn.microsoft.com/azure/ai-foundry/model-inference/overview?WT.mc_id=academic-105485-koreyst) 來原型化 AI 模型的相關說明。

模型變體：
- Llama 3.1 - 70B 指令型
- Llama 3.1 - 405B 指令型
- Llama 3.2 - 11B 視覺指令型
- Llama 3.2 - 90B 視覺指令型

*注意：Llama 3 亦可於 Microsoft Foundry Models 取得，但本課程不涵蓋該模型*



## Llama 3.1 

Llama 3.1 以 4050 億參數，屬於開源大型語言模型（LLM）類別。 

此版本是對早期發布的 Llama 3 的升級，提供： 

- 更大的上下文窗口 - 128k 字元對比 8k 字元 
- 更大的最大輸出字元數量 - 4096 對比 2048 
- 更佳的多語言支援 - 由於訓練字元數增加 

這些使 Llama 3.1 能夠處理更複雜的生成式 AI 應用案例，包括： 
- 原生函數調用 - 能夠調用 LLM 工作流程外的外部工具與函數
- 更好的檢索增強生成（RAG）性能 - 由於更大的上下文窗口 
- 合成數據生成 - 能夠為微調等任務創建有效數據 



### 原生函數呼叫

Llama 3.1 已經被微調得更有效於執行函數或工具的呼叫。它亦有兩個內置工具，模型可以根據用戶的提示識別需要使用的工具。這些工具是：

- **Brave Search** - 可用於通過網絡搜索獲取最新資訊，如天氣
- **Wolfram Alpha** - 可用於更複雜的數學計算，因此不需要自己撰寫函數。

你亦可以創建自己的自定義工具供 LLM 呼叫。

在以下代碼示例中：

- 我們在系統提示中定義可用工具（brave_search、wolfram_alpha）。
- 傳送一個詢問某城市天氣的用戶提示。
- LLM 將回應對 Brave Search 工具的呼叫，類似這樣 `<|python_tag|>brave_search.call(query="Stockholm weather")`

*注意：此示例僅做工具呼叫，如果你想取得結果，你需在 Brave API 頁面創建免費賬戶並自行定義函數。 


In [ ]:
%pip install azure-core
%pip install azure-ai-inference

In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import AssistantMessage, SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]
model_name = "Meta-Llama-3.1-405B-Instruct"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


tool_prompt=f"""
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Environment: ipython
Tools: brave_search, wolfram_alpha
Cutting Knowledge Date: December 2023
Today Date: 23 July 2024

You are a helpful assistant<|eot_id|>
"""

messages = [
    SystemMessage(content=tool_prompt),
    UserMessage(content="What is the weather in Stockholm?"),

]

response = client.complete(messages=messages, model=model_name)

print(response.choices[0].message.content)






### Llama 3.2 

儘管係一個大型語言模型，Llama 3.1 嘅其中一個限制就係多模態能力。即係能夠使用唔同類型嘅輸入，例如用圖像作提示並提供回應。呢個能力係 Llama 3.2 嘅主要功能之一。呢啲功能亦包括：

- 多模態 - 有能力評估文字同圖像提示
- 小至中型變體（11B 同 90B）- 提供靈活嘅部署選擇，
- 純文字變體（1B 同 3B）- 使模型能夠部署喺邊緣／流動裝置，並提供低延遲

呢個多模態支援係開源模型領域一大步。以下嘅代碼示例會同時使用圖像同文字提示，從 Llama 3.2 90B 取得圖像分析結果。

### 使用 Llama 3.2 多模態支援


In [ ]:
%pip install azure-core
%pip install azure-ai-inference

In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import (
    SystemMessage,
    UserMessage,
    TextContentItem,
    ImageContentItem,
    ImageUrl,
    ImageDetailLevel,
)
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]
model_name = "Llama-3.2-90B-Vision-Instruct"


client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)

response = client.complete(
    messages=[
        SystemMessage(
            content="You are a helpful assistant that describes images in details."
        ),
        UserMessage(
            content=[
                TextContentItem(text="What's in this image?"),
                ImageContentItem(
                    image_url=ImageUrl.load(
                        image_file="sample.jpg",
                        image_format="jpg",
                        detail=ImageDetailLevel.LOW)
                ),
            ],
        ),
    ],
    model=model_name,
)

print(response.choices[0].message.content)


## 學習不止於此，繼續你的旅程

完成本課程後，請瀏覽我們的[生成式人工智能學習合集](https://aka.ms/genai-collection?WT.mc_id=academic-105485-koreyst)，繼續提升你的生成式人工智能知識！


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們力求準確，但請注意，自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議尋求專業人工翻譯。我們不對因使用本翻譯而引起的任何誤解或曲解承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
